In [1]:
import requests
import pandas as pd

OTP_URL = "http://localhost:8080/otp/gtfs/v1"

def query_otp(query: str, variables: dict | None = None) -> dict:
    """Run a GraphQL query against the OTP server and return the `data` payload."""
    resp = requests.post(
        OTP_URL,
        json={"query": query, "variables": variables or {}},
        headers={"Content-Type": "application/json"},
        timeout=60,
    )
    resp.raise_for_status()
    payload = resp.json()
    if "errors" in payload:
        raise RuntimeError(payload["errors"])
    return payload["data"]

### `planConnection` mode toggles (`modes: PlanModesInput`)

Confirmed via introspection against the local OTP 2.6.0 server (`/otp/gtfs/v1`). `planConnection` is the only field with this granularity — the stable `plan` field only takes a combined `transportModes: [{mode, qualifier}]` list.

| Variable | GraphQL field | Controls | Enum values |
|---|---|---|---|
| `ACCESS_MODES` | `modes.transit.access` | Origin → first transit stop | `WALK`, `BICYCLE`, `BICYCLE_PARKING`, `BICYCLE_RENTAL`, `CAR_PARKING`, `CAR_DROP_OFF`, `CAR_RENTAL`, `SCOOTER_RENTAL`, `FLEX` |
| `EGRESS_MODES` | `modes.transit.egress` | Last transit stop → destination | `WALK`, `BICYCLE`, `BICYCLE_RENTAL`, `CAR_PICKUP`, `CAR_RENTAL`, `SCOOTER_RENTAL`, `FLEX` |
| `TRANSFER_MODES` | `modes.transit.transfer` | Between transit legs | `WALK`, `BICYCLE` |
| — (not currently toggled) | `modes.direct` | Whole-journey mode for a non-transit itinerary | `WALK`, `BICYCLE`, `BICYCLE_PARKING`, `BICYCLE_RENTAL`, `CAR`, `CAR_PARKING`, `CAR_RENTAL`, `SCOOTER_RENTAL`, `FLEX` |
| `TRANSIT_ONLY` | `modes.transitOnly` | If `True`, skip direct (non-transit) itineraries entirely | `Boolean` |
| — (not currently toggled) | `modes.directOnly` | If `True`, skip transit search entirely, only return direct itineraries | `Boolean` |
| — (not currently toggled) | `modes.transit.transit` | Which transit modes are usable, e.g. `[{"mode": "BUS"}]` — omit to allow all | `TransitMode` enum: `AIRPLANE`, `BUS`, `CABLE_CAR`, `CARPOOL`, `COACH`, `FERRY`, `FUNICULAR`, `GONDOLA`, `MONORAIL`, `RAIL`, `SUBWAY`, `TAXI`, `TRAM`, `TROLLEYBUS` |

Notes:
- All `transit.*` mode lists default to `[WALK]` if omitted — you don't have to set them explicitly.
- `direct` and `transit` fields are independent: `directOnly`/`transitOnly` decide which *kind* of itinerary comes back, while `direct`/`transit.access`/etc. decide *which modes* are allowed within that kind.
- `planConnection` is marked `@deprecated(reason: "Experimental and can include breaking changes, use plan instead")` in OTP's schema — re-check these field names after upgrading the OTP jar.


In [1]:
PLAN_QUERY = """
query Plan(
  $origin: PlanLabeledLocationInput!
  $destination: PlanLabeledLocationInput!
  $dateTime: PlanDateTimeInput
  $modes: PlanModesInput
) {
  planConnection(
    origin: $origin
    destination: $destination
    dateTime: $dateTime
    modes: $modes
  ) {
    edges {
      node {
        start
        end
        duration
        numberOfTransfers
        legs {
          mode
          distance
          duration
          start { scheduledTime }
          end { scheduledTime }
          route { shortName longName }
          from { name }
          to { name }
          fareProducts {
            product {
              name
              ... on DefaultFareProduct {
                price { amount currency { code } }
              }
            }
          }
        }
      }
    }
  }
}
"""

# Toggle these to change access/egress/transfer/direct behavior.
# See PlanAccessMode / PlanEgressMode / PlanTransferMode / PlanDirectMode enums.
ACCESS_MODES = []         # origin -> first transit stop (not selected)
EGRESS_MODES = []         # last transit stop -> destination (not selected)
TRANSFER_MODES = []       # between transit legs (not selected)
TRANSIT_ONLY = False      # include both direct and transit itineraries (not selected)

variables = {
    "origin": {"location": {"coordinate": {"latitude": -37.81, "longitude": 144.96}}},      # Melbourne
    "destination": {"location": {"coordinate": {"latitude": -33.8688, "longitude": 151.2093}}},  # Sydney
    "dateTime": {"earliestDeparture": "2026-05-20T08:00:00+10:00"},
    "modes": {
        "transitOnly": TRANSIT_ONLY,
        "transit": {
            "access": ACCESS_MODES,
            "egress": EGRESS_MODES,
            "transfer": TRANSFER_MODES,
        },
    },
}

data = query_otp(PLAN_QUERY, variables)
itineraries = [edge["node"] for edge in data["planConnection"]["edges"]]
print(f"{len(itineraries)} itineraries found")

NameError: name 'query_otp' is not defined

In [ ]:
def format_duration(seconds: float | None) -> str | None:
    """e.g. 5025 -> '1h 23m', 180 -> '3m'"""
    if seconds is None:
        return None
    total_min = round(seconds / 60)
    h, m = divmod(total_min, 60)
    return f"{h}h {m:02d}m" if h else f"{m}m"


def leg_fare(leg: dict) -> tuple[float | None, str | None]:
    for fp in leg["fareProducts"]:
        price = fp["product"].get("price")
        if price:
            return price["amount"], price["currency"]["code"]
    return None, None


# Flatten legs across all itineraries into a readable DataFrame
rows = []
for i, itin in enumerate(itineraries):
    for leg in itin["legs"]:
        fare_amount, fare_currency = leg_fare(leg)
        rows.append({
            "itinerary": i,
            "mode": leg["mode"],
            "route": leg["route"]["shortName"] if leg["route"] else None,
            "from": leg["from"]["name"],
            "to": leg["to"]["name"],
            "depart": leg["start"]["scheduledTime"],
            "arrive": leg["end"]["scheduledTime"],
            "duration": format_duration(leg["duration"]),
            "distance_km": round(leg["distance"] / 1000, 2) if leg["distance"] else None,
            "fare": fare_amount,
            "currency": fare_currency,
        })

legs_df = pd.DataFrame(rows)
legs_df["depart"] = pd.to_datetime(legs_df["depart"]).dt.strftime("%H:%M")
legs_df["arrive"] = pd.to_datetime(legs_df["arrive"]).dt.strftime("%H:%M")
legs_df

In [ ]:
# Aggregate each itinerary into a single "route" row: total cost, distance, time
route_rows = []
for i, itin in enumerate(itineraries):
    legs = itin["legs"]
    transit_legs = [leg for leg in legs if leg["route"] is not None]
    modes_used = list(dict.fromkeys(leg["mode"] for leg in transit_legs))  # unique, in order

    total_distance_km = sum(leg["distance"] for leg in legs if leg["distance"]) / 1000

    total_fare = 0.0
    currency = None
    for leg in legs:
        amount, cur = leg_fare(leg)
        if amount:
            total_fare += amount
            currency = currency or cur

    route_rows.append({
        "itinerary": i,
        "origin": legs[0]["from"]["name"],
        "destination": legs[-1]["to"]["name"],
        "depart": itin["start"],
        "arrive": itin["end"],
        "duration": format_duration(itin["duration"]),
        "transfers": itin["numberOfTransfers"],
        "modes": " -> ".join(modes_used) if modes_used else "WALK",
        "distance_km": round(total_distance_km, 2),
        "total_fare": round(total_fare, 2) if currency else None,
        "currency": currency,
    })

routes_df = pd.DataFrame(route_rows)
routes_df["depart"] = pd.to_datetime(routes_df["depart"]).dt.strftime("%Y-%m-%d %H:%M")
routes_df["arrive"] = pd.to_datetime(routes_df["arrive"]).dt.strftime("%Y-%m-%d %H:%M")
routes_df